In [41]:
import pandas as pd
import numpy as np

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from umap import UMAP

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Chargement des données

In [42]:
counts = pd.read_csv(
    "counts.tsv",
    sep="\t",
    index_col=0
)

print("Matrice brute :", counts.shape)

Matrice brute : (41130, 40)


# 2. Filtrage des gènes peu exprimés

In [43]:
MIN_COUNTS = 10
MIN_SAMPLE_FRAC = 0.20

min_samples = max(
    2,
    int(np.ceil(MIN_SAMPLE_FRAC * counts.shape[1]))
)

keep = (
    (counts >= MIN_COUNTS)
    .sum(axis=1)
    >= min_samples
)

# PyDESeq2 expects int -> also cast all columns
counts_filt = counts.loc[keep].map(int)

print(
    f"Gènes après filtrage : "
    f"{counts_filt.shape[0]}"
)

Gènes après filtrage : 25696


# 3. Préparation pour PyDESeq2

In [44]:
# PyDESeq2 attend :
# samples x genes

counts_for_deseq = counts_filt.T

metadata = pd.DataFrame(
    {
        "condition": ["all"] * counts_for_deseq.shape[0]
    },
    index=counts_for_deseq.index
)

# 4. Estimation des size factors DESeq2

In [45]:
inference = DefaultInference(n_cpus=8)

dds = DeseqDataSet(
    counts=counts_for_deseq,
    metadata=metadata,
    design="~condition",
    refit_cooks=True,
    inference=inference,
    quiet=False,
)

dds.fit_size_factors()


# comptes normalisés DESeq2
#norm_counts = dds.layers["normed_counts"]

# BETTER: Use pyDESeq2's VST (Variance Stabilizing Transformation)
dds.vst()
norm_counts = dds.layers["vst_counts"]

norm_counts = pd.DataFrame(
    norm_counts,
    index=counts_for_deseq.index,
    columns=counts_for_deseq.columns
)

print(
    "Matrice normalisée :",
    norm_counts.shape
)

Using None as control genes, passed at DeseqDataSet initialization
Fit type used for VST : parametric
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.06 seconds.

Fitting size factors...
... done in 0.05 seconds.

Fitting dispersions...
... done in 6.37 seconds.



Matrice normalisée : (40, 25696)


# 5. Batch effect correction / transformation

In [46]:
USE_COMBAT = False

if USE_COMBAT:
    from combat.pycombat import pycombat  # pip install combat

    # Use file describing batches (1 by row)
    datasets = []
    with open('batches.tsv') as batches_file:
        for batch_line in batches_file:
            datasets.append(batch_line.rstrip('\n').split('\t'))

    batch = []
    for j in range(len(datasets)):
        batch.extend([j for _ in range(len(datasets[j]))])
    
    # Then run (py)combat:
    log_expr = pycombat(norm_counts.T, batch).T

else:
    # OLD: log-transform
    #log_expr = np.log2(norm_counts + 1)

    # BETTER: Use pyDESeq2's VST (Variance Stabilizing Transformation)
    log_expr = norm_counts

# 6. Sélection des gènes les plus variables

In [47]:
N_VARIABLE_GENES = 5000

gene_variance = log_expr.var(axis=0)

n_keep = min(
    N_VARIABLE_GENES,
    len(gene_variance)
)

top_genes = (
    gene_variance
    .nlargest(n_keep)
    .index
)

expr_var = log_expr[top_genes]

print(
    f"Gènes variables conservés : "
    f"{expr_var.shape[1]}"
)

Gènes variables conservés : 5000


# 7. Standardisation

In [48]:
# Afterwards Copilot says not needed
# Especially if run K-means on PCA res

#expr_scaled = (
#    expr_var
#    - expr_var.mean(axis=0)
#) / (
#    expr_var.std(axis=0) + 1e-8
#)

expr_scaled = expr_var

# Write filt+norm file:
expr_scaled.T.to_csv("counts_norm.tsv", sep="\t")
print("Wrote: 'counts_norm.tsv'")

Wrote: 'counts_norm.tsv'
